In [ ]:
%matplotlib widget
import numpy as np
import tomllib
from pathlib import Path

from helpers.plotting import plot_performance, plot_gradients, plot_weight_matrices, InteractiveSpikePlot

In [ ]:
work_dir = Path.cwd()
print(work_dir)
with open(work_dir / "config.toml", 'rb') as cfg_file:
    cfg = tomllib.load(cfg_file)

n_trials = cfg["training"]["n_trials"]

def load_recording(data_name: str, key: str, skip_is: list[int] = []) -> np.ndarray:
    data = []
    for i in range(n_trials):
        if i not in skip_is:
            with np.load(work_dir / f"seed_{i}" / "data_net" / f"{data_name}.npz") as d:
                data.append(d[key])
    return np.array(data)

def get_data(data_name: str, skip_is: list[int] = []) -> dict[np.ndarray]:
    with np.load(work_dir / f"seed_0" / "data_net" / f"{data_name}.npz") as d:
        keys = list(d.keys())
    return {key: load_recording(data_name, key, skip_is) for key in keys}

def get_data_seed(data_name: str, seed: int) -> dict[np.ndarray]:
    with np.load(work_dir / f"seed_{seed}" / "data_net" / f"{data_name}.npz") as d:
        return {key: d[key] for key in list(d.keys())}

def filter_seeds(data: np.ndarray, seeds: list[int]) -> np.ndarray:
    return np.take(data, seeds, axis=0)

def unpack_bits(data: dict[np.ndarray], key: str, suffix: str = "-n_cols") -> np.ndarray:
    unpack_key = key + suffix
    if unpack_key in data.keys():
        return np.unpackbits(data[key], axis=-1, count=int(data[unpack_key])).astype(bool)
    return data[key]

In [ ]:
skip_is = []

In [ ]:
plot_seeds = []
perf_data = get_data("performance", skip_is)
print(np.isnan(perf_data["loss"]))
plot_performance(filter_seeds(perf_data["loss"], plot_seeds), filter_seeds(perf_data["acc"], plot_seeds))

In [ ]:
max_grad_rec = get_data("gradients", "avg_grad")
avg_grad_rec = get_data("gradients", "max_grad")
plot_gradients(max_grad_rec, avg_grad_rec)

In [ ]:
weights_forward_0 = get_data("weights", "weights_forward_0")
plot_weight_matrices(weights_forward_0, name="weights_forward_0")

weights_forward_1 = get_data("weights", "weights_forward_1")
plot_weight_matrices(weights_forward_1, name="weights_forward_1")

weights_recurrent_0 = get_data("weights", "weights_recurrent_0")
plot_weight_matrices(weights_recurrent_0, name="weights_recurrent_0")

In [ ]:
layer_spikes = {
    "spikes_in": get_data("spikes", "spikes_in")[0],
    "spikes_hidden_0": get_data("spikes", "spikes_hidden_0")[0],
    "spikes_out": get_data("spikes", "spikes_out")[0],
    "targets": get_data("spikes", "targets")[0],
}
isp = InteractiveSpikePlot(layer_spikes, range(1, 21))
isp.draw()
# isp.save_as_pdf()